<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Описание задачи: 
Создать базовый класс PaymentMethod в C#, который будет представлять 
различные способы оплаты. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов должны быть реализованы новые атрибуты и методы, а также 
переопределены некоторые методы базового класса для демонстрации 
полиморфизма. 
Требования к базовому классу PaymentMethod: 

• Атрибуты: 
1. ID способа оплаты (PaymentMethodId)
2. Название способа оплаты (MethodName)
3. Минимальная сумма (MinAmount). 

• Методы: 
1. ProcessPayment(decimal amount): метод для обработки платежа 
указанной суммы. 
2. CheckMinimumAmount(decimal amount): метод для проверки 
минимальной суммы платежа. 
3. GetPaymentDetails(): метод для получения деталей способа оплаты.
 
Требования к производным классам: 
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные 
атрибуты, 
такие 
как 
URL платежной системы (PaymentUrl). 
Метод ProcessPayment() должен быть переопределен для включения URL 
платежной системы в процесс оплаты. 
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные 
атрибуты, 
такие 
как 
Банковские 
данные 
(BankData). 
Метод CheckMinimumAmount() должен быть переопределен для проверки 
минимальной суммы платежа с учетом банковских комиссий. 
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать 
дополнительные атрибуты, такие как Место выдачи наличных 
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен 
для отображения места выдачи наличных.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;

public interface IRefundable
{
    bool Refund(decimal amount);
}

public interface ICancelable
{
    void Cancel();
}

public interface ILogger
{
    void Log(string message);
}

public class ConsoleLogger : ILogger
{
    public void Log(string message) => Console.WriteLine($"[ЛОГ] {message}");
}

public delegate void TransactionHandler(string methodName, decimal amount, bool isSuccess);

public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    protected string _currency;
    protected bool _isEnabled;
    protected int _transactionCount;
    protected string _providerName;
    protected DateTime _lastTransactionDate;
    protected bool _requiresKyc;
    protected string _securityLevel;

    public List<string> SupportedCurrencies { get; } = new List<string>();
    public List<decimal> TransactionHistory { get; } = new List<decimal>();
    public string CurrentStatus { get; private set; } = "Активен";

    public event TransactionHandler OnTransactionCompleted;

    public string ProviderName { get { return _providerName; } set { _providerName = value; } }
    public DateTime LastTransactionDate { get { return _lastTransactionDate; } set { _lastTransactionDate = value; } }
    public int PaymentMethodId { get { return _paymentMethodId; } set { _paymentMethodId = value; } }
    public string MethodName { get { return _methodName; } set { _methodName = value; } }
    public decimal MinAmount { get { return _minAmount; } set { if (value >= 0) _minAmount = value; } }
    public string Currency { get { return _currency; } set { _currency = value; } }
    public bool IsEnabled { get { return _isEnabled; } set { _isEnabled = value; } }
    public bool RequiresKyc { get { return _requiresKyc; } set { _requiresKyc = value; } }
    public string SecurityLevel { get { return _securityLevel; } set { _securityLevel = value; } }

    public PaymentMethod(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel)
    {
        this.PaymentMethodId = id;
        this.MethodName = name; 
        this.MinAmount = minAmount;
        this.Currency = currency;
        this.ProviderName = provider;
        this.RequiresKyc = requiresKyc;
        this.SecurityLevel = securityLevel;
        this.IsEnabled = true;
        this._transactionCount = 0;
        this._lastTransactionDate = DateTime.MinValue;
        
        SupportedCurrencies.Add(currency);
    }

    public virtual void ProcessPayment(decimal amount)
    {
        if (!IsEnabled) 
        { 
            CurrentStatus = "Отключен";
            Console.WriteLine($"Метод '{MethodName}' отключен."); 
            OnTransactionCompleted?.Invoke(MethodName, amount, false);
            return; 
        }
        
        if (CheckMinimumAmount(amount))
        {
            _transactionCount++;
            _lastTransactionDate = DateTime.Now;
            TransactionHistory.Add(amount);
            CurrentStatus = "Успешно";
            Console.WriteLine($"[УСПЕХ] Платеж через {MethodName} ({ProviderName}) на {amount} {Currency} обработан.");
            OnTransactionCompleted?.Invoke(MethodName, amount, true);
        }
        else
        {
            CurrentStatus = "Отклонен";
            Console.WriteLine($"[ОТКАЗ] Платеж на {amount} {Currency} отклонен. Мин. сумма: {MinAmount} {Currency}.");
            OnTransactionCompleted?.Invoke(MethodName, amount, false);
        }
    }

    public virtual void ProcessPayment(decimal amount, string promoCode)
    {
        Console.WriteLine($"[ИНФО] Получен запрос на оплату {amount} {Currency} с промокодом '{promoCode}'.");
        decimal discountedAmount = amount * 0.9m;
        Console.WriteLine($"[ИНФО] Итоговая сумма к оплате после скидки: {discountedAmount} {Currency}.");
        ProcessPayment(discountedAmount);
    }

    public virtual bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;

    public int GetTransactionCount() => _transactionCount;
    public void Disable() => IsEnabled = false;

    public void AddSupportedCurrency(string currency) 
    { 
        if (!SupportedCurrencies.Contains(currency)) SupportedCurrencies.Add(currency); 
    }
    
    public decimal GetAverageTransactionAmount() 
    { 
        return TransactionHistory.Count > 0 ? TransactionHistory.Average() : 0; 
    }
    
    public void ClearHistory() => TransactionHistory.Clear();
}

public class OnlinePayment : PaymentMethod, IRefundable, ICancelable
{
    private string _paymentUrl;
    private decimal _transactionFee;
    private int _maxDailyTransactions;
    private bool _is3DSecureEnabled;

    public string DeviceId { get; set; }
    public bool IsTwoFactorEnabled { get; set; }
    public string LastLoginIp { get; set; }

    public string PaymentUrl { get { return _paymentUrl; } set { _paymentUrl = value; } }
    public int MaxDailyTransactions { get { return _maxDailyTransactions; } set { _maxDailyTransactions = value; } }

    public OnlinePayment(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, string url, decimal fee, int maxTrans, string deviceId, string ip) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel)
    {
        this.PaymentUrl = url;
        this._transactionFee = fee;
        this.MaxDailyTransactions = maxTrans;
        this.DeviceId = deviceId;
        this.LastLoginIp = ip;
        this.IsTwoFactorEnabled = true;
    }

    public override void ProcessPayment(decimal amount)
    {
        if (GetTransactionCount() >= MaxDailyTransactions) 
        { 
            Console.WriteLine($"[ОТКАЗ] Превышен дневной лимит операций ({MaxDailyTransactions})."); 
            return; 
        }
        base.ProcessPayment(amount);
        Console.WriteLine($"--> Деньги списаны по URL: {PaymentUrl} (Комиссия: {_transactionFee} {Currency})");
    }

    public bool Refund(decimal amount) 
    { 
        Console.WriteLine($"[ВОЗВРАТ] {amount} {Currency} возвращены на карту."); 
        return true; 
    }

    void ICancelable.Cancel() => Console.WriteLine($"[ОТМЕНА] Онлайн-платеж {MethodName} отменен на стороне шлюза.");

    public void Enable2FA() { IsTwoFactorEnabled = true; Console.WriteLine("2FA включена."); }
    public void Disable2FA() { IsTwoFactorEnabled = false; Console.WriteLine("2FA отключена."); }
    public void CheckDeviceTrust() { Console.WriteLine($"Проверка устройства {DeviceId} с IP {LastLoginIp}: Доверенное."); }
}

public class BankTransfer : PaymentMethod, ICancelable
{
    private decimal _bankData;
    private string _swiftCode;
    private string _correspondentAccount;
    private int _processingTimeHours;

    public string BeneficiaryName { get; set; }
    public string AccountNumber { get; set; }
    public bool IsPriority { get; set; }

    public decimal BankData { get { return _bankData; } set { _bankData = value; } }
    public string SwiftCode { get { return _swiftCode; } set { _swiftCode = value; } }

    public BankTransfer(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, decimal bankData, string swift, string corrAcc, int time, string beneficiary, string account, bool isPriority) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel)
    {
        this.BankData = bankData;
        this._swiftCode = swift;
        this._correspondentAccount = corrAcc;
        this._processingTimeHours = time;
        this.BeneficiaryName = beneficiary;
        this.AccountNumber = account;
        this.IsPriority = isPriority;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount / 100 * BankData;
        decimal amountAfterCommission = amount - commission;
        if (amountAfterCommission >= MinAmount) return true;
        Console.WriteLine($"[ОТКАЗ] Недостаточно средств. Комиссия: {commission}. Остаток: {amountAfterCommission} (нужно {MinAmount})");
        return false;
    }

    void ICancelable.Cancel() => Console.WriteLine($"[ОТМЕНА] Банковский перевод {MethodName} отменен.");

    public void SetPriority() { IsPriority = true; Console.WriteLine("Перевод помечен как срочный."); }
    public bool ValidateAccount() { return AccountNumber.Length >= 10; }
    public void GetTransferDetails() { Console.WriteLine($"Получатель: {BeneficiaryName}, Счет: {AccountNumber}, Срочный: {IsPriority}"); }
}

public class CashPayment : PaymentMethod, ICancelable
{
    private string _cashPickupPoint;
    private string _operatorName;
    private string _cashierName;
    private decimal _cashInDrawer;
    public string TerminalId { get; set; }
    public string Location { get; set; }
    public bool AcceptsCoins { get; set; }

    public string CashierName { get { return _cashierName; } set { _cashierName = value; } }
    public decimal CashInDrawer { get { return _cashInDrawer; } set { _cashInDrawer = value; } }

    public CashPayment(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, string pickupPoint, string opName, string cashier, decimal cash, string terminalId, string location, bool acceptsCoins) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel)
    {
        this._cashPickupPoint = pickupPoint;
        this._operatorName = opName;
        this._cashierName = cashier;
        this._cashInDrawer = cash;
        this.TerminalId = terminalId;
        this.Location = location;
        this.AcceptsCoins = acceptsCoins;
    }

    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount); 
        _cashInDrawer += amount;
        Console.WriteLine($"--> Деньги приняты. Теперь у кассира {_cashierName}: {_cashInDrawer} {Currency}");
    }

    void ICancelable.Cancel() => Console.WriteLine($"[ОТМЕНА] Наличная операция {MethodName} отменена, чек аннулирован.");

    public void SetLocation(string newLocation) { Location = newLocation; Console.WriteLine($"Локация терминала {TerminalId} обновлена: {Location}"); }
    public void ToggleCoinAcceptance() { AcceptsCoins = !AcceptsCoins; Console.WriteLine($"Прием монет: {(AcceptsCoins ? "ВКЛ" : "ВЫКЛ")}"); }
    public void GetTerminalStatus() { Console.WriteLine($"Терминал {TerminalId} в {Location}. Монеты: {(AcceptsCoins ? "Да" : "Нет")}"); }
}

public class CryptoPayment : OnlinePayment, ICancelable
{
    private double _networkFee;
    private int _confirmationsRequired;

    public string WalletAddress { get; set; }
    public string NetworkName { get; set; }
    public double CurrentGasPrice { get; set; }

    public double NetworkFee { get { return _networkFee; } set { _networkFee = value; } }
    public int ConfirmationsRequired { get { return _confirmationsRequired; } set { _confirmationsRequired = value; } }

    public CryptoPayment(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, string url, decimal fee, int maxTrans, string deviceId, string ip, double netFee, int conf, string wallet, string network, double gas) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel, url, fee, maxTrans, deviceId, ip)
    {
        this._networkFee = netFee;
        this._confirmationsRequired = conf;
        this.WalletAddress = wallet;
        this.NetworkName = network;
        this.CurrentGasPrice = gas;
    }

    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"[КРИПТО] Попытка оплаты {amount} {Currency}");
        decimal totalAmount = amount + (decimal)_networkFee;
        Console.WriteLine($"[КРИПТО] Итоговая сумма к списанию: {totalAmount} {Currency} (Gas: {CurrentGasPrice})");
        base.ProcessPayment(totalAmount); 
    }

    void ICancelable.Cancel() => Console.WriteLine($"[ОТМЕНА] Крипто-транзакция {MethodName} отменена.");

    public void SwitchNetwork(string newNetwork) { NetworkName = newNetwork; Console.WriteLine($"Сеть переключена на {NetworkName}"); }
    public void UpdateGasPrice(double newGas) { CurrentGasPrice = newGas; Console.WriteLine($"Gas обновлен: {CurrentGasPrice}"); }
    public void GetWalletInfo() { Console.WriteLine($"Кошелек: {WalletAddress}, Сеть: {NetworkName}"); }
}

public class PaymentTerminal<T> where T : PaymentMethod
{
    private List<T> _availableMethods = new List<T>();
    private readonly ILogger _logger;
    
    public Dictionary<string, int> UsageStatistics { get; } = new Dictionary<string, int>();

    public PaymentTerminal(ILogger logger)
    {
        _logger = logger;
    }

    public void AddMethod(T method) 
    { 
        _availableMethods.Add(method); 
        UsageStatistics[method.MethodName] = 0;
        _logger.Log($"Метод {method.MethodName} добавлен в терминал.");
    }

    public void ProcessAllAvailable(decimal amount)
    {
        _logger.Log($"Запуск обработки платежа на сумму {amount} через все методы.");
        foreach (var method in _availableMethods) 
        { 
            method.ProcessPayment(amount);
            UsageStatistics[method.MethodName]++;
        }
    }

    public void PrintStatistics()
    {
        Console.WriteLine("\n--- Статистика использования методов ---");
        foreach (var stat in UsageStatistics)
        {
            Console.WriteLine($"{stat.Key}: использован {stat.Value} раз(а)");
        }
    }
}

public class Customer
{
    public string Name { get; set; }
    private decimal _balance;
    private string _loyaltyTier;
    private readonly ILogger _logger;

    public List<PaymentMethod> SavedMethods { get; } = new List<PaymentMethod>();

    public decimal Balance { get { return _balance; } set { _balance = value; } }
    public string LoyaltyTier { get { return _loyaltyTier; } set { _loyaltyTier = value; } }

    public Customer(string name, decimal balance, string tier, ILogger logger)
    {
        Name = name;
        _balance = balance;
        _loyaltyTier = tier;
        _logger = logger;
    }

    public void MakePurchase(PaymentMethod method, decimal amount)
    {
        _logger.Log($"Клиент {Name} инициирует платеж на {amount} {method.Currency}.");
        method.ProcessPayment(amount);
    }

    public void RequestRefund(IRefundable refundableMethod, decimal amount)
    {
        _logger.Log($"Клиент {Name} запрашивает возврат {amount}.");
        refundableMethod.Refund(amount);
    }

    public void AddToWallet(PaymentMethod method) 
    { 
        SavedMethods.Add(method); 
        Console.WriteLine($"[КОШЕЛЕК] Метод {method.MethodName} сохранен.");
    }
    
    public void RemoveFromWallet(PaymentMethod method) 
    { 
        SavedMethods.Remove(method); 
        Console.WriteLine($"[КОШЕЛЕК] Метод {method.MethodName} удален.");
    }
    
    public void ShowWallet()
    {
        Console.WriteLine($"\n[КОШЕЛЕК {Name}] Сохраненные методы ({SavedMethods.Count}):");
        foreach (var method in SavedMethods)
        {
            Console.WriteLine($"- {method.MethodName} ({method.ProviderName})");
        }
    }
}

ILogger logger = new ConsoleLogger();

BankTransfer bank = new BankTransfer(2, "Банковский перевод", 50, "RUB", "Сбербанк", true, "Высокий", 5, "SABRRUMM", "40702810...", 24, "ООО Ромашка", "40702810123456789012", false);
CashPayment cash = new CashPayment(3, "Наличные", 100, "RUB", "Газпромбанк", false, "Средний", "Касса №1", "Ольга", "Лариса", 5000, "TERM-001", "Москва, ул. Ленина 1", true);
OnlinePayment online = new OnlinePayment(1, "Онлайн оплата", 10, "RUB", "ЮKassa", true, "Высокий", "https://pay.ru", 2, 10, "DEV-999", "192.168.1.1");
CryptoPayment crypto = new CryptoPayment(4, "Крипто-оплата", 500, "USDT", "Binance", false, "Средний", "https://crypto.gate", 1, 5, "WAL-123", "10.0.0.5", 0.5, 3, "0x123...abc", "Ethereum", 25.5);

Customer buyer = new Customer("Кирилл", 10000, "Gold", logger);
PaymentTerminal<PaymentMethod> terminal = new PaymentTerminal<PaymentMethod>(logger);

Console.WriteLine("=== 1. Демонстрация новых методов классов ===");
bank.GetTransferDetails();
bank.SetPriority();
cash.GetTerminalStatus();
crypto.GetWalletInfo();
crypto.UpdateGasPrice(30.0);

Console.WriteLine("\n=== 2. Демонстрация коллекций (Кошелек клиента) ===");
buyer.AddToWallet(online);
buyer.AddToWallet(crypto);
buyer.ShowWallet();
buyer.RemoveFromWallet(cash);
buyer.ShowWallet();

Console.WriteLine("\n=== 3. Демонстрация событий и делегатов ===");
online.OnTransactionCompleted += (methodName, amount, isSuccess) => 
{
    Console.WriteLine($"[СОБЫТИЕ] Транзакция '{methodName}' на сумму {amount} завершена. Статус: {(isSuccess ? "УСПЕХ" : "ОТКАЗ")}");
};

bank.OnTransactionCompleted += (methodName, amount, isSuccess) => 
{
    Console.WriteLine($"[СОБЫТИЕ] Транзакция '{methodName}' на сумму {amount} завершена. Статус: {(isSuccess ? "УСПЕХ" : "ОТКАЗ")}");
};

Console.WriteLine("\nВызов платежей (события сработают автоматически):");
buyer.MakePurchase(online, 1000);
buyer.MakePurchase(bank, 200);

Console.WriteLine("\n=== 4. Демонстрация коллекций в Терминале (Словарь статистики) ===");
terminal.AddMethod(bank);
terminal.AddMethod(cash);
terminal.AddMethod(crypto);

terminal.ProcessAllAvailable(500);
terminal.PrintStatistics();

Console.WriteLine("\n=== 5. Проверка истории транзакций и средних значений ===");
Console.WriteLine($"Средний чек онлайн-оплаты: {online.GetAverageTransactionAmount()} RUB");
Console.WriteLine($"История онлайн-оплаты: {string.Join(", ", online.TransactionHistory)}");

=== 1. Демонстрация новых методов классов ===
Получатель: ООО Ромашка, Счет: 40702810123456789012, Срочный: False
Перевод помечен как срочный.
Терминал TERM-001 в Москва, ул. Ленина 1. Монеты: Да
Кошелек: 0x123...abc, Сеть: Ethereum
Gas обновлен: 30

=== 2. Демонстрация коллекций (Кошелек клиента) ===
[КОШЕЛЕК] Метод Онлайн оплата сохранен.
[КОШЕЛЕК] Метод Крипто-оплата сохранен.

[КОШЕЛЕК Кирилл] Сохраненные методы (2):
- Онлайн оплата (ЮKassa)
- Крипто-оплата (Binance)
[КОШЕЛЕК] Метод Наличные удален.

[КОШЕЛЕК Кирилл] Сохраненные методы (2):
- Онлайн оплата (ЮKassa)
- Крипто-оплата (Binance)

=== 3. Демонстрация событий и делегатов ===

Вызов платежей (события сработают автоматически):
[ЛОГ] Клиент Кирилл инициирует платеж на 1000 RUB.
[УСПЕХ] Платеж через Онлайн оплата (ЮKassa) на 1000 RUB обработан.
[СОБЫТИЕ] Транзакция 'Онлайн оплата' на сумму 1000 завершена. Статус: УСПЕХ
--> Деньги списаны по URL: https://pay.ru (Комиссия: 2 RUB)
[ЛОГ] Клиент Кирилл инициирует платеж на 200 RUB